# Canopy Watch | Rondônia radar case
Run this notebook from the project root or `pipeline/`. In Colab, add the unzipped project to the runtime and set the working directory to its root. The optional setup cell installs geospatial dependencies on a Debian/Ubuntu runtime. The browser app uses the generated files directly.


In [ ]:
# Colab/Linux only, if system GDAL or SciPy is missing. Uncomment to run.
# !apt-get update -qq && apt-get install -y -qq gdal-bin python3-gdal python3-numpy python3-scipy


In [ ]:
from pathlib import Path
import json, subprocess

ROOT = Path.cwd()
if ROOT.name == "pipeline":
    ROOT = ROOT.parent
assert (ROOT / "pipeline/rondonia.json").exists(), "Open the notebook from the Canopy Watch project."
PYTHON = "/usr/bin/python3"
manifest = ROOT / "pipeline/rondonia.json"
cache = ROOT / "cache"
assets = ROOT / "dist/data"


## Acquire and align the observations
The stored manifest pins five compatible Sentinel-1A radar scenes, two Sentinel-2 optical scenes and the DEM. GDAL crops the inputs onto a common UTM grid. This step needs internet access.


In [ ]:
subprocess.run([PYTHON, str(ROOT / "pipeline/fetch_case.py"), str(manifest), str(cache)], check=True)


## Compute first detections and compare the footprints
The detector applies a -3.5 dB VH threshold to valid forest-proxy cells and keeps contiguous alerts of at least 0.96 ha. It records the first qualifying date separately for every cell. The featured patch is selected from persistent detections with strong agreement with the annual tree-cover-loss reference. The selected stable footprint serves as a local comparison.


In [ ]:
subprocess.run([PYTHON, str(ROOT / "pipeline/build_case.py"), str(cache), str(manifest), str(assets)], check=True)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

case = json.loads((assets / "case.json").read_text())
dates = [scene["date"] for scene in case["radar"]]
fig, ax = plt.subplots(figsize=(9, 3.5))
for site in case["patches"]:
    ax.plot(dates, [item["delta"] for item in site["series"]], marker="o", label=site["title"])
ax.axhline(case["threshold"], color="gray", linestyle="--", label="Per-cell threshold")
ax.set_ylabel("Mean VH change (dB)")
ax.set_title("Featured footprint and stable forest comparison")
ax.legend()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
patch = case["patches"][0]
x, y = patch["x"] * case["width"], patch["y"] * case["height"]
for i, ax in enumerate(axes):
    ax.imshow(mpimg.imread(assets / f"optical-{i}.png"))
    ax.imshow(mpimg.imread(assets / "outline.png"))
    ax.set(xlim=(x-105,x+105), ylim=(y+75,y-75), title=case["optical"][i]["date"])
    ax.axis("off")
plt.show()


## Interpretation
The threshold operates per cell. The chart shows an average over fixed footprints. The selected patch and stable comparison illustrate this case and do not provide an independent dataset for accuracy estimation. Source links and the exact processing steps are in the project README and pipeline.
